<a href="https://colab.research.google.com/github/janani26121992/AI-Projects/blob/main/AI_LSTM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Problem Statement**

# “To develop a deep learning model using LSTM that can generate meaningful and coherent text in the style of Shakespeare by learning patterns from a given text dataset.”

# **Detailed Problem Statement**

# *The objective of this project is to build an LSTM-based text generation model that learns from a Shakespeare dataset and generates new text sequences word-by-word. The model should produce text that is grammatically meaningful, contextually relevant, and stylistically similar to the training data.*

# **Necessary Libraries**

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# **Data Gathering**
"I used the requests library to fetch the Shakespeare dataset directly from GitHub using a URL. Then I converted the response into text format and previewed the first 500 characters."

In [2]:
pip install datasets

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("guslovesmath/shakespeare-plays-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'shakespeare-plays-dataset' dataset.
Path to dataset files: /kaggle/input/shakespeare-plays-dataset


In [4]:
import pandas as pd
import os

# Check files inside folder
print(os.listdir(path))

# Load CSV file
file_path = os.path.join(path, "shakespeare_plays.csv")
df = pd.read_csv(file_path)

print(df.head())

['shakespeare_plays.csv']
   Unnamed: 0                  play_name   genre character  act  scene  \
0           0  All's Well That Ends Well  Comedy  Countess    1      1   
1           1  All's Well That Ends Well  Comedy   Bertram    1      1   
2           2  All's Well That Ends Well  Comedy   Bertram    1      1   
3           3  All's Well That Ends Well  Comedy   Bertram    1      1   
4           4  All's Well That Ends Well  Comedy     Lafeu    1      1   

   sentence                                               text     sex  
0         1  In delivering my son from me, I bury a second ...  female  
1         2  And I in going, madam, weep o'er my father's d...    male  
2         3  anew: but I must attend his majesty's command, to    male  
3         4     whom I am now in ward, evermore in subjection.    male  
4         5  You shall find of the king a husband, madam; you,    male  


In [5]:
print(df.columns)

Index(['Unnamed: 0', 'play_name', 'genre', 'character', 'act', 'scene',
       'sentence', 'text', 'sex'],
      dtype='object')


In [6]:
text_data = " ".join(df['text'].dropna())

# Preview
print(text_data[:500])

In delivering my son from me, I bury a second husband. And I in going, madam, weep o'er my father's death anew: but I must attend his majesty's command, to whom I am now in ward, evermore in subjection. You shall find of the king a husband, madam; you, sir, a father: he that so generally is at all times good must of necessity hold his virtue to you; whose worthiness would stir it up where it wanted rather than lack it where there is such abundance. What hope is there of his majesty's amendment? 


In [7]:
import re
text_data = text_data.lower()
text_data = re.sub(r'[^a-zA-Z\s]', '', text_data)

In [8]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text_data])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary Size:", total_words)

# Reverse mapping: index -> word
index_word = {i: word for word, i in tokenizer.word_index.items()}

Vocabulary Size: 27588


In [9]:
token_list = tokenizer.texts_to_sequences([text_data])[0]

input_sequences = []

for i in range(8, len(token_list)):
    n_gram_sequence = token_list[i-5:i+1]
    input_sequences.append(n_gram_sequence)

print(input_sequences[:5])

[[161, 48, 13, 3, 1793, 7], [48, 13, 3, 1793, 7, 794], [13, 3, 1793, 7, 794, 318], [3, 1793, 7, 794, 318, 2], [1793, 7, 794, 318, 2, 3]]


In [10]:
max_sequence_length = 9

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1]
Y = input_sequences[:, -1]

# Convert target to one-hot encoding
Y = to_categorical(Y, num_classes=total_words)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (811297, 5)
Y shape: (811297, 27588)


In [11]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequence_length - 1,)))

# Embedding layer
model.add(Embedding(input_dim=total_words, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(total_words, activation='softmax')) #units=6032

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 8, 128)         │     3,531,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 8, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 27588)          │     2,786,388 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,595,552 (25.16 MB)

 Trainable params: 6,595,552 (25.16 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,Y,epochs=40,batch_size=32,verbose=1,callbacks=[early_stop])

In [ ]:
model.save("TextGenerationModel1.keras")

In [ ]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]

    # index of top k proabilities : [index]
    top_probs = preds[top_indices]

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

In [ ]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length - 1,
            padding='pre'
        )

        predicted_probs = model.predict(token_list, verbose=0)[0] # [[0.89,0.07,.....,]] = [0.89,0.07,.....,]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.8,
            top_k=5
        )

        next_word = index_word.get(predicted_index, "")

        # avoid immediate repetition
        if next_word in generated_words[-3:]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

In [ ]:
print(generate_text("my lord", next_words=10))

In [ ]:
print(generate_text("the king shall", next_words=10))